In [ ]:
# =====================================================================#  SISTEMA LINEALIZADO  —  Polonyi, arXiv:1701.04068v4, ecs. (7)-(15)##  Autocontenido: solo numpy / scipy / matplotlib. No importa nada del repo.##  Convenciones:  signatura (+,-,-,-),  c = 1,  r0 = 1.#  El cutoff ell se mide en unidades del radio clasico, de modo que el eje#  de la Fig. 2 del paper es r0/ell.# =====================================================================import numpy as npimport matplotlib.pyplot as pltfrom scipy.optimize import newtonfrom scipy.special import roots_genlaguerrer0 = 1.0plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,                     "grid.alpha": .3, "font.size": 10})

In [ ]:
# --- REGULADORES -----------------------------------------------------# El regulador ES la no-localidad: sustituye la delta de la funcion de Green# retardada, delta(x^2) -> delta_B(x^2), sujeta a tres condiciones (p. 6):#   (i)   int dz delta_B(z) = 1      preserva el flujo radiado#   (ii)  delta_B(0) = 0             separa los puntos singulares#   (iii) delta_B(z) = 0 si z < 0    SUPRIME LA INTERACCION SUPERLUMINICA# La (iii) es el mecanismo entero: una trayectoria runaway tendria que# superar c para escapar, y ahi el regulador apaga la autointeraccion.## Todo el bloque linealizado necesita del regulador una sola cosa: la# cuadratura de  int_{-inf}^{0} du delta_B(u^2) g(u)  como nodos y pesos.def nodos_pesos(kind, ell, n=120):    """(u, w) con int_{-inf}^0 du delta_B(u^2) g(u) = sum_i w_i g(u_i)."""    if kind == "desplazado":                       # ec. (4): delta(x^2 - ell^2)        # delta(u^2-ell^2) = [d(u+ell)+d(u-ell)]/(2 ell); sobre u<0 solo u=-ell        return np.array([-ell]), np.array([1.0 / (2 * ell)])    if kind == "suavizado":                        # ec. (5)        # delta_B(z) = Theta(z) z exp(-sqrt(z)/ell) / (12 ell^4).        # Sustituyendo u = -ell t queda Gauss-Laguerre generalizada (alpha=2).        t, w = roots_genlaguerre(n, 2.0)        return -ell * t, w / (12 * ell)    raise ValueError(kind)def delta_m_sobre_m(kind, ell):    """delta_m/m de la ec. (10). Forma cerrada, derivada a mano."""    return r0 / (2 * ell) if kind == "desplazado" else r0 / (6 * ell)def m_sobre_mB(kind, ell):    """m/m_B = 1/(1 - delta_m/m). Linea punteada de la Fig. 2 del paper."""    return 1.0 / (1.0 - delta_m_sobre_m(kind, ell))

In [ ]:
# --- NUMERADOR DE LA EC. (14) ---------------------------------------#   N(phi) = (1 + i phi - phi^2) e^{-i phi} - 1 + phi^2/2 - (2/3) i phi^3## Los cuatro primeros ordenes se cancelan IDENTICAMENTE, dejando#       N(phi) = sum_{n>=4} (-i phi)^n (n-1)^2 / n!# (se comprueba con sum (n-1)^2 x^n/n! = (x^2 - x + 1) e^x,  x = -i phi).# El orden dominante es (3/8) phi^4, que coincide con lo que afirma el# paper en la p. 8: esa es la verificacion de que la ec. (14) esta bien# transcrita.## La forma DIRECTA es inutilizable a phi pequeno: la cancelacion destruye# toda la precision. Por eso la serie.def N(phi, corte=6.0, terminos=64):    phi = np.atleast_1d(np.asarray(phi, dtype=complex))    out = np.empty_like(phi)    chico = np.abs(phi) < corte    if chico.any():        x = -1j * phi[chico]        a, acc = np.ones_like(x), np.zeros_like(x)        for n in range(1, terminos):            a = a * x / n            if n >= 4:                acc += a * (n - 1) ** 2        out[chico] = acc    if (~chico).any():        p = phi[~chico]        with np.errstate(over="ignore", invalid="ignore"):            out[~chico] = ((1 + 1j*p - p**2) * np.exp(-1j*p)                           - 1 + p**2/2 - (2/3)*1j*p**3)    return outdef chi(omega, kind, ell):    """Susceptibilidad chi^r_omega, ec. (14). Vectorizada sobre omega complejo.    F^r_omega = 1/[(omega+i eps)^2 chi^r_omega], asi que los POLOS de F^r son    los CEROS de chi. Estable y causal <=> sin ceros en Im(omega) > 0.    """    omega = np.atleast_1d(np.asarray(omega, dtype=complex))    u, w = nodos_pesos(kind, ell)    with np.errstate(invalid="ignore", divide="ignore"):        integ = np.einsum("j,ij->i", w, N(omega[:, None] * u[None, :]) / u**2)        br = (2/3)*1j*omega - 2*integ/omega**2    br = np.where(np.abs(omega) < 1e-12, 0j, br)     # omega -> 0: chi -> 1    return 1.0 + r0 * br

In [ ]:
# --- VERIFICACION ----------------------------------------------------# Contra formas cerradas, no contra otra corrida del propio codigo.ell = 0.4# 1) Momentos.  I2 = int du delta_B(u^2) u^2  fija el orden omega^2 de chi.for kind, i2_exacto in (("suavizado", 2*ell), ("desplazado", ell/2)):    u, w = nodos_pesos(kind, ell)    print(f"{kind:11s}  I2 = {np.sum(w*u**2):.10f}   exacto {i2_exacto:.10f}"          f"   |   delta_m/m = {delta_m_sobre_m(kind, ell):.6f}")# 2) Orden dominante de N.p = 1e-3print(f"\nN({p}) = {N(p)[0].real:.6e}   (3/8)phi^4 = {(3/8)*p**4:.6e}")# 3) Para el regulador desplazado la integral es EXACTA, asi que chi tiene#    forma cerrada. Es el test mas fuerte de la cuadratura del suavizado.w_ = np.array([0.2, 1.0, 3.0, 8.0 + 0.3j])cerrada = 1 + r0*((2/3)*1j*w_ - N(-w_*ell)/(w_**2 * ell**3))print(f"\nchi desplazado vs forma cerrada: {np.max(np.abs(chi(w_,'desplazado',ell)-cerrada)):.2e}")# 4) Contraste con el paquete `nlaid` del repo, que tiene 50 tests. A#    r0/ell = 3 su cero dominante esta documentado en +2.573621 -0.717930i.#    Este notebook lo reproduce bit a bit; si tocas la cuadratura y este#    numero se mueve, sabras que rompiste algo.z = newton(lambda z: chi(z, "suavizado", 1/3)[0], 2.5-0.7j, tol=1e-13, maxiter=80)print(f"cero dominante a r0/ell=3: {z.real:+.6f} {z.imag:+.6f} i"      f"   (referencia +2.573621 -0.717930 i)")

In [ ]:
# --- CEROS DE chi EN EL SEMIPLANO SUPERIOR ---------------------------# Dos herramientas que se contrastan entre si:#   - principio del argumento: cuenta vueltas de chi alrededor del origen#     sobre un contorno cerrado. Global, no depende de semillas.#   - scipy.optimize.newton: localiza cada cero (acepta complejos).def cuenta_ceros(kind, ell, re_max=12., im_lo=1e-6, im_hi=12., n=800):    lados = [np.linspace(-re_max, re_max, n) + 1j*im_lo,             re_max + 1j*np.linspace(im_lo, im_hi, n),             np.linspace(re_max, -re_max, n) + 1j*im_hi,             -re_max + 1j*np.linspace(im_hi, im_lo, n)]    c = np.concatenate(lados + [lados[0][:1]])    return int(round(np.sum(np.diff(np.unwrap(np.angle(chi(c, kind, ell)))))/(2*np.pi)))def busca_ceros(kind, ell, re_max=12., im_lo=1e-4, im_hi=12., malla=140):    """Minimos locales de |chi| como semillas + refinamiento por newton."""    RE, IM = np.meshgrid(np.linspace(-re_max, re_max, malla),                         np.linspace(im_lo, im_hi, malla//2), indexing="ij")    W = RE + 1j*IM    A = np.abs(chi(W.ravel(), kind, ell)).reshape(W.shape)    ceros = []    for i in range(1, A.shape[0]-1):        for j in range(1, A.shape[1]-1):            if A[i, j] > A[i-1:i+2, j-1:j+2].min():                continue            try:                # errstate acotado: la secante de scipy divide por cero cuando                # se estanca, y ese iterado se descarta igualmente.                with np.errstate(invalid="ignore", divide="ignore"):                    z = newton(lambda z: chi(z, kind, ell)[0], W[i, j],                               tol=1e-12, maxiter=60)            except (RuntimeError, OverflowError, ValueError):                continue            if (im_lo < z.imag < im_hi and abs(z.real) < re_max                    and abs(chi(z, kind, ell)[0]) < 1e-8                    and not any(abs(z-q) < 1e-6 for q in ceros)):                ceros.append(complex(z))    return sorted(ceros, key=lambda z: -z.imag)

In [ ]:
# --- EL POLO RUNAWAY DE ABRAHAM-LORENTZ ------------------------------# Al remover el cutoff, chi -> 1 + (2/3) i r0 omega  (ec. 15), cuyo cero# esta en omega = 3i/2: SEMIPLANO SUPERIOR, la autoaceleracion.ell_p = 0.02print("principio del argumento:", cuenta_ceros("suavizado", ell_p, re_max=6, im_hi=6))for z in busca_ceros("suavizado", ell_p, re_max=6, im_hi=6, malla=120):    print(f"   omega = {z.real:+.5f} {z.imag:+.5f} i      (ec. 15 predice 1.5i)")

In [ ]:
# --- BARRIDO EN ell --------------------------------------------------# Modifica esta linea para explorar:ELLS = np.array([2.0, 1.0, 0.5, 0.3, 0.25, 0.2, 0.15])KIND = "suavizado"def modo_dominante(kind, ell):    """Cero de chi que gobierna la dinamica linealizada.    Si hay ceros en el semiplano SUPERIOR devuelve el de mayor Im (el modo    inestable). Si no, busca en el INFERIOR el menos amortiguado, que es el    que fija la relajacion.    """    arriba = busca_ceros(kind, ell, re_max=8, im_hi=8, malla=110)    if arriba:        return arriba[0]    RE, IM = np.meshgrid(np.linspace(-8, 8, 130),                         np.linspace(max(-0.97/ell, -8), -0.02, 70), indexing="ij")    W = RE + 1j*IM    A = np.abs(chi(W.ravel(), kind, ell)).reshape(W.shape)    sem = [W[i, j] for i in range(1, A.shape[0]-1) for j in range(1, A.shape[1]-1)           if A[i, j] <= A[i-1:i+2, j-1:j+2].min()]    for z0 in sorted(sem, key=lambda z: -z.imag):        try:            with np.errstate(invalid="ignore", divide="ignore"):                z = newton(lambda z: chi(z, kind, ell)[0], z0, tol=1e-12, maxiter=60)        except (RuntimeError, OverflowError, ValueError):            continue        if z.imag < 0 and abs(chi(z, kind, ell)[0]) < 1e-8:            return complex(z)    return np.nanprint(f"{'ell':>6} {'r0/ell':>7} {'delta_m/m':>10} {'m_B/m':>8} "      f"{'Re w':>9} {'Im w':>9}  regimen")modos = {}for L in ELLS:    z = modo_dominante(KIND, L)    modos[L] = z    dm = delta_m_sobre_m(KIND, L)    reg = "INESTABLE" if (np.isfinite(z) and z.imag > 0) else "relaja"    print(f"{L:6.2f} {1/L:7.2f} {dm:10.4f} {1-dm:8.4f} "          f"{z.real:+9.4f} {z.imag:+9.4f}  {reg}")

In [ ]:
# --- TRAYECTORIA DEL SISTEMA LINEALIZADO -----------------------------# La ec. (7) es LINEAL: tras apagar la fuente externa, la solucion es una# superposicion de modos e^{-i omega_k s} sobre los ceros de chi. A tiempos# largos manda el dominante, asi que  xddot ~ Re[A e^{-i w s}]  y las dos# integrales son inmediatas -- no hace falta ningun integrador:#       xdot = Re[A e^{-i w s} / (-i w)]#       x    = Re[A e^{-i w s} / (-i w)^2]def modo(z, s, A=1.0):    e = A * np.exp(-1j * z * s)    return e.real, (e/(-1j*z)).real, (e/(-1j*z)**2).real   # xddot, xdot, xs = np.linspace(0, 12, 2000)RAMPA = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]from matplotlib.colors import LinearSegmentedColormapcol = LinearSegmentedColormap.from_list("r", RAMPA)(np.linspace(0, 1, len(ELLS)))fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))for L, c in zip(ELLS, col):    z = modos[L]    if not np.isfinite(z):        continue    a, v, x = modo(z, s)    lab = f"$r_0/\\ell$={1/L:.2f}"    ax[0].plot(s, x, lw=1.5, color=c, label=lab)    ax[1].semilogy(s, np.abs(a) + 1e-16, lw=1.5, color=c, label=lab)ax[0].set_xlabel("$s/r_0$"); ax[0].set_ylabel("$x(s)$  (modo dominante)")ax[0].set_title("A.  Trayectoria linealizada", loc="left", fontsize=10.5)ax[0].set_ylim(-3, 3); ax[0].legend(fontsize=8, frameon=False)ax[1].set_xlabel("$s/r_0$"); ax[1].set_ylabel(r"$|\ddot{x}|$")ax[1].set_title("B.  Aceleración (log)", loc="left", fontsize=10.5)ax[1].set_ylim(1e-8, 1e3); ax[1].legend(fontsize=8, frameon=False)xx = np.linspace(0.05, 8, 400)ax[2].plot(xx, [1 - delta_m_sobre_m("suavizado", 1/q) for q in xx],           lw=2, color="#5c5b54", label="suavizado ec. (5)")ax[2].plot(xx, [1 - delta_m_sobre_m("desplazado", 1/q) for q in xx],           lw=2, ls="--", color="#5c5b54", label="desplazado ec. (4)")for L, c in zip(ELLS, col):    ax[2].plot([1/L], [1 - delta_m_sobre_m(KIND, L)], "o", ms=8, color=c,               mec="white", mew=1.5, zorder=5)ax[2].axhline(0, color="#b0afa8", lw=1.2)ax[2].set_xlabel("$r_0/\\ell$"); ax[2].set_ylabel("$m_B/m = 1-\\delta m/m$")ax[2].set_title("C.  Masa desnuda, ec. (10)", loc="left", fontsize=10.5)ax[2].set_ylim(-1.2, 1.1); ax[2].legend(fontsize=8, frameon=False)plt.tight_layout(); plt.show()

In [ ]:
# --- CUTOFF CRITICO --------------------------------------------------# Donde el primer cero entra al semiplano superior. Biseccion sobre r0/ell.def cutoff_critico(kind, lo=0.3, hi=30., tol=5e-3):    f = lambda q: cuenta_ceros(kind, 1/q, re_max=8, im_hi=8, n=600) > 0    assert not f(lo) and f(hi), "la raiz no esta acotada"    while hi - lo > tol:        mid = (lo + hi) / 2        lo, hi = (lo, mid) if f(mid) else (mid, hi)    return (lo + hi) / 2for kind in ("desplazado", "suavizado"):    q = cutoff_critico(kind)    print(f"{kind:11s}  r0/ell critico = {q:.3f}   (ell_c = {1/q:.3f} r0)")